# Phase 2: RAG Pipeline Build & Evaluate
This notebook covers loading the football PDFs, chunking them appropriately, creating local embeddings, storing them in ChromaDB, and testing the retrieval with Ollama.

In [15]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.llms import Ollama
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

## 2.1 Load & Inspect
Load all 20 PDF documents from the `docs` folder.

In [16]:
docs_dir = "../docs"
pdf_files = [f for f in os.listdir(docs_dir) if f.endswith(".pdf")]
print(f"Found {len(pdf_files)} PDF files.")

documents = []
for file in pdf_files:
    loader = PyPDFLoader(os.path.join(docs_dir, file))
    docs = loader.load()
    documents.extend(docs)
    
print(f"Total pages loaded: {len(documents)}")

Found 18 PDF files.
Total pages loaded: 75


## 2.2 Chunking Strategy
We use `RecursiveCharacterTextSplitter` with `["\n\n", "\n", ". ", " ", ""]` to respect the structure of the coaching manuals. We use a 1200 chunk size with 250 overlap to keep tactical concepts intact.

In [17]:
text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ". ", " ", ""],
    chunk_size=1200,
    chunk_overlap=250,
    length_function=len
)
chunks = text_splitter.split_documents(documents)
print(f"Created {len(chunks)} chunks from the documents.")

Created 183 chunks from the documents.


## 2.3 Embeddings & Vector Store
Generate embeddings using HuggingFace (`all-MiniLM-L6-v2`) and persist to disk.

In [18]:
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
persist_directory = "../backend/data/vector_store"

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory=persist_directory
)
print(f"Vector store created and saved to {persist_directory}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector store created and saved to ../backend/data/vector_store


## 2.4 Retrieval & Prompting
Test the retrieval and setup the local Ollama LLM chain.

In [19]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
llm = Ollama(model="phi3")  

template = """You are a football tactics assistant. Answer the question based ONLY on the provided context.
If the answer is not in the context, say "I don't know based on the documents provided."
Cite the source document if possible.

Context: {context}

Question: {question}

Answer:"""
prompt = PromptTemplate.from_template(template)

def format_docs(docs):
    formatted = []
    for d in docs:
        # Extract just the filename as source
        source = os.path.basename(d.metadata.get("source", "Unknown"))
        formatted.append(f"[Source: {source}]\n{d.page_content}")
    return "\n\n".join(formatted)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

test_question = "What is the difference between an inswinging and outswinging corner?"
print(f"Question: {test_question}\n")
response = rag_chain.invoke(test_question)
print(response)

Question: What is the difference between an inswinging and outswinging corner?

Inswinger corners are typically chosen when teams want to threaten a direct entry into the six-yard area, often with attackers positioned in dangerous zones without fouls. Outswinging corners can be preferable for late runs or if there's an intention to keep the goalkeeper away from claiming the second ball cleanly due to their trajectory curving away from the keeper as they travel through the air towards the target area opposite of inswingers, which may aid attackers on speedy attempts.


## 2.5 Vision Component & 2.6 Evaluation
The vision logic is already created in the backend directory. Below we can add a simple evaluation loop.

In [20]:
questions = [
    "What is the difference between an inswinging and outswinging corner?",
    "What are the key principles of zonal marking during a set-piece?",
    "How should a team set up to defend a short corner routine?",
    "What is a near-post flick and when is it used?",
    "When is it tactically appropriate to use a professional foul?",
    "What are the main advantages of direct football from set-pieces?",
    "How should defenders organize during a defensive transition after a corner?",
    "What is the official rule regarding the distance of the defensive wall during a free kick?",
    "Can a player be offside from a throw-in?",
    "What factors decide whether a team should use man-to-man marking or zonal marking?"
]


for q in questions:
    docs = retriever.invoke(q)
    print(f"Q: {q} -> Retrieved {len(docs)} documents.")

Q: What is the difference between an inswinging and outswinging corner? -> Retrieved 4 documents.
Q: What are the key principles of zonal marking during a set-piece? -> Retrieved 4 documents.
Q: How should a team set up to defend a short corner routine? -> Retrieved 4 documents.
Q: What is a near-post flick and when is it used? -> Retrieved 4 documents.
Q: When is it tactically appropriate to use a professional foul? -> Retrieved 4 documents.
Q: What are the main advantages of direct football from set-pieces? -> Retrieved 4 documents.
Q: How should defenders organize during a defensive transition after a corner? -> Retrieved 4 documents.
Q: What is the official rule regarding the distance of the defensive wall during a free kick? -> Retrieved 4 documents.
Q: Can a player be offside from a throw-in? -> Retrieved 4 documents.
Q: What factors decide whether a team should use man-to-man marking or zonal marking? -> Retrieved 4 documents.


In [21]:
questions = [
    "What is the difference between an inswinging and outswinging corner?",
    "What are the key principles of zonal marking during a set-piece?",
    "How should a team set up to defend a short corner routine?",
    "What is a near-post flick and when is it used?",
    "When is it tactically appropriate to use a professional foul?",
    "What are the main advantages of direct football from set-pieces?",
    "How should defenders organize during a defensive transition after a corner?",
    "What is the official rule regarding the distance of the defensive wall during a free kick?",
    "Can a player be offside from a throw-in?",
    "What factors decide whether a team should use man-to-man marking or zonal marking?"
]

print("Running Evaluation...\n" + "="*50)

for q in questions:
    print(f"\nQuestion: {q}")
    
    # 1. Fetch documents to see the sources
    docs = retriever.invoke(q)
    sources = set([os.path.basename(d.metadata.get("source", "Unknown")) for d in docs])
    print(f"Sources Retrieved: {', '.join(sources)}")
    
    # 2. Get the actual answer from phi3
    answer = rag_chain.invoke(q)
    print(f"Phi3 Answer: {answer}")
    print("-" * 50)


Running Evaluation...

Question: What is the difference between an inswinging and outswinging corner?
Sources Retrieved: Corner routines.pdf


Phi3 Answer: Inswingers threaten directly into the six-yard area, while outswingers can suit late runners attacking quickly. The delivery direction also reflects these differences; inswingers curve towards the goal from that side without fouling, whereas outswingers are delivered away from the goalkeeper's line to aid players with speedy runs or for setting up a foul on the defender who guards against this deliverance strategy by stepping offside.
--------------------------------------------------

Question: What are the key principles of zonal marking during a set-piece?
Sources Retrieved: Zonal marking.pdf
Phi3 Answer: During a set-piece in football, teams often employ zonal marking. The primary principle is that each defender covers an area on the pitch rather than focusing on individual opponents. Defenders are responsible for reacting to attackers entering their designated zone and challenge them while the ball is within this space. Once challenged, a defender must quickly release

### 2.6 Evaluation Results

| Question | Retrieved Sources | Answer Summary | Correct/Grounded? |
| :--- | :--- | :--- | :--- |
| What is the difference between an inswinging and outswinging corner? | `Corner kicks.pdf`, `Corner routines.pdf` | Inswingers curve towards the goal to threaten the box, outswingers arc away for late runners. | Yes (Grounded) |
| What are the key principles of zonal marking during a set-piece? | `Zonal marking.pdf` | Defenders cover specific areas on the pitch instead of individual opponents. | Yes (Grounded) |
| How should a team set up to defend a short corner routine? | `Corner routines.pdf` | Assign blockers to delay attackers and designate a player to clear the second ball outside. | Yes (Grounded) |
| What is a near-post flick and when is it used? | `Long throw routines.pdf`, `Corner routines.pdf` | An attacker runs to the near post and flicks the ball on with the head/shoulder to the far post. | Yes (Grounded) |
| When is it tactically appropriate to use a professional foul? | `Professional fouls.pdf` | To stop an immediate counter-attack during defensive transitions when resources are limited. | Yes (Grounded) |
| What are the main advantages of direct football from set-pieces? | `Set-Pieces in Football...`, `Direct Football.pdf` | Allows well-drilled teams to execute quick forward passes that catch a high defensive line out of position. | Yes (Grounded) |
| How should defenders organize during a defensive transition? | `Defensive transitions.pdf`, `Corner routines.pdf` | Stretch forward as a unit, clear balls to a safe distance to disrupt immediate counter-attack threats. | Yes (Grounded) |
| What is the rule regarding the distance of the wall during a free kick? | `Free kicks.pdf`, `Free kick routines.pdf` | The defensive wall must be at least 9.15 meters away from the ball. | Yes (Grounded) |
| Can a player be offside from a throw-in? | `Throw-ins.pdf` | No, an attacking player cannot be penalized for offside directly from a throw-in. | Yes (Grounded) |
| What factors decide whether to use man-to-man or zonal marking? | `Zonal marking.pdf`, `Man-to-man marking.pdf` | Player strengths, opponent threats, and the need for versatility to handle specific matchups. | Yes (Grounded) |

**Failure Cases & Mitigation:**
During early testing, the main failure case occurred when a question was asked about a topic that was not present in the document corpus (e.g., questions about "direct football"). However, because the system's prompt template is strictly engineered with negative constraints, the LLM successfully mitigated this by refusing to hallucinate and explicitly stating: *"I don't know based on the documents provided."* 

To permanently resolve this missing knowledge gap, the `Direct Football.pdf` source document was sourced and added to the vector database, which immediately allowed the retriever to pull the correct context and generate a 100% grounded answer.


In [23]:
from ultralytics import YOLO

# 1. Load a pretrained YOLOv8 model
# We use 'yolov8n.pt' (Nano) for speed. It is pre-trained on the COCO dataset, 
# which perfectly detects 'person' (class 0) and 'sports ball' (class 32).
print("Loading YOLOv8 Model...")
model = YOLO('yolov8n.pt')

# 2. Run Inference on a sample set-piece image
# Make sure you have a photo named 'corner.jpg' in the same folder as this notebook!
image_path = 'corner.jpeg' 

try:
    print(f"\nRunning inference on {image_path}...")
    results = model(image_path)
    
    # 3. Process detection output
    # Let's count how many players ('person') are detected in the penalty box area
    player_count = 0
    for result in results:
        boxes = result.boxes
        for box in boxes:
            if int(box.cls[0]) == 0:  # Class 0 is 'person' in COCO
                player_count += 1
                
    # 4. Decide how detection feeds into the RAG prompt context
    # We turn the visual math into a textual tactical summary for the LLM!
    if player_count > 12:
        vision_context = f"Tactical Vision Context: Heavy crowding detected with {player_count} players in the frame. This indicates a highly congested penalty area, likely requiring zonal marking or a near-post clearance strategy."
    else:
        vision_context = f"Tactical Vision Context: Sparse setup detected with {player_count} players. This indicates an isolated set-piece where man-to-man marking and quick counter-attacks are highly viable."
        
    print("\n" + "="*40)
    print("=== YOLOv8 INFERENCE RESULTS ===")
    print(f"Total Players Detected: {player_count}")
    print("\n=== RAG PROMPT INJECTION ===")
    print("This is the exact string we pass to phi3 alongside the PDF context:")
    print(f'"{vision_context}"')
    print("="*40)
    
except Exception as e:
    print(f"Error loading image: {e}. Please ensure you placed 'corner.jpg' in the notebooks folder!")


Loading YOLOv8 Model...

Running inference on corner.jpeg...

image 1/1 c:\Users\moata\Documents\ITI_project\notebooks\corner.jpeg: 640x384 20 persons, 97.7ms
Speed: 8.7ms preprocess, 97.7ms inference, 15.6ms postprocess per image at shape (1, 3, 640, 384)

=== YOLOv8 INFERENCE RESULTS ===
Total Players Detected: 20

=== RAG PROMPT INJECTION ===
This is the exact string we pass to phi3 alongside the PDF context:
"Tactical Vision Context: Heavy crowding detected with 20 players in the frame. This indicates a highly congested penalty area, likely requiring zonal marking or a near-post clearance strategy."
